# Práctica 02 — Texto bruto y tokenización

**Curso:** NLP Clásico con Python · **Clase 02** · **Tiempo estimado:** 60 minutos
**Notebook de clase:** `notebooks_clase/clase02_texto_crudo_tokenizacion.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los textos y los resultados esperados están en la celda de datos que sigue al setup. Son cortos y están escritos para el curso (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. No hace falta ninguna librería aparte de Python (usamos `re`, `unicodedata` y `collections`).

---
## Celda 0 — Setup y datos

In [ ]:
import re
import unicodedata
from collections import Counter

print("Librerías cargadas correctamente.")

In [ ]:
# datos-es
PALABRA_NFC = "canción"
PALABRA_NFD = "cancio" + "́" + "n"

CASOS_LIMPIEZA = [
    ("¡Qué   bueno!!!  https://t.co/x1", "¡Qué bueno!"),
    ("@tienda Llegó tardeeee...", "Llegó tarde..."),
    ("Cuesta 1000 euros", "Cuesta 1000 euros"),
    ("Non è  vero ,  davvero???", "Non è vero, davvero?"),
    ("Fantastico #consigliato www.sito.it", "Fantastico"),
]

CASOS_TOKEN = [
    ("¿Dónde está l'uomo?", ["¿", "Dónde", "está", "l'", "uomo", "?"]),
    ("Costa 4,99 euro.", ["Costa", "4,99", "euro", "."]),
    ("Ho un'ora, dell'amico.", ["Ho", "un'", "ora", ",", "dell'", "amico", "."]),
    ("Vamos al mercado, ¡ya!", ["Vamos", "al", "mercado", ",", "¡", "ya", "!"]),
    ("Il 12.5 costa 1.250,50 euro", ["Il", "12.5", "costa", "1.250,50", "euro"]),
]

TOKENS_ES = "el sol sale y el sol se pone y el día pasa".split()
TOKENS_IT = "il gatto dorme e il cane dorme ma il topo no".split()

PARES = [
    ("BUEN PRODUCTO, LLEGÓ RÁPIDO", "Buen producto, llegó rápido"),
    ("Muy bueno!!  https://t.co/a1", "Muy bueno!"),
    ("Nos gustó mucho 😍", "Nos gustó mucho"),
    ("El envio llego tarde", "El envío llegó tarde"),
    ("Es un desatre total", "Es un desastre total"),
    ("Todo perfecto", "Todo perfecto"),
    ("Me encantó   el diseño", "Me encantó el diseño"),
    ("Precio justo, pero la pantalla se raya facil", "Precio justo, pero la pantalla se raya fácil"),
    ("Llegó a tiempo. Gracias @soporte", "Llegó a tiempo. Gracias"),
    ("Recomendable 100%", "Recomendable 100%"),
    ("OTTIMO SERVIZIO", "Ottimo servizio"),
    ("Consegna veloce   e imballo perfetto", "Consegna veloce e imballo perfetto"),
    ("Non e' male, ma nn lo ricomprerei", "Non è male, ma non lo ricomprerei"),
    ("Pacco arrivato rotto 😡 https://goo.gl/x", "Pacco arrivato rotto"),
    ("Bellissimooo, lo consiglio", "Bellissimo, lo consiglio"),
    ("Servizio scadente. Mai piu", "Servizio scadente. Mai più"),
    ("Ottima qualità, prezzo onesto", "Ottima qualità, prezzo onesto"),
    ("Troppo caro per quello che offre!!!", "Troppo caro per quello che offre!"),
    ("un po' lento ma funziona", "un po' lento ma funziona"),
    ("Delusa dall'assistenza  #delusione", "Delusa dall'assistenza"),
]

ABREVIATURAS = {"es": {"q": "que", "xq": "porque", "pq": "porque", "tb": "también"},
                "it": {"nn": "non", "ke": "che", "cmq": "comunque", "xke": "perché"}}

PRUEBA_RETO = [
    ("es", "@tienda ¡¡LLEGÓÓÓ ya!!  q bueno, cuesta 4,99 € 😍 https://t.co/x9",
     ["llegó", "ya", "que", "bueno", "cuesta", "4,99"]),
    ("es", "Es _muy_ malo, xq no funciona   ....",
     ["es", "muy", "malo", "porque", "no", "funciona"]),
    ("es", "¿Dónde está el pedidooo del Sr. Ruiz? #urgente",
     ["dónde", "está", "el", "pedido", "del", "sr", "ruiz"]),
    ("es", "Pagué 1.250,50 euros y 3,5 % de comisión 😡😡",
     ["pagué", "1.250,50", "euros", "y", "3,5", "de", "comisión"]),
    ("it", "L'ordine nn è arrivato, cmq l'assistenza risponde!!! 👍",
     ["l'", "ordine", "non", "è", "arrivato", "comunque", "l'", "assistenza", "risponde"]),
    ("it", "BELLISSIMOOO, un'ora d'attesa 😍 www.sito.it/r/12",
     ["bellissimo", "un'", "ora", "d'", "attesa"]),
    ("it", "Il _Resegone_ costa 12.5 euro, ke bello   @negozio",
     ["il", "resegone", "costa", "12.5", "euro", "che", "bello"]),
    ("it", "dell'amico  Prezzo:   4,99!!!! Perfettoooo #consigliato",
     ["dell'", "amico", "prezzo", "4,99", "perfetto"]),
]

## Ejercicio 1 — NFC, NFD y longitud

`PALABRA_NFC` (`canción`) está escrita con la `ó` como **un** punto de código; `PALABRA_NFD` es la misma palabra con la `o` y un acento combinante. Calcula con `unicodedata`:

- `cadena_nfd`: `PALABRA_NFC` convertida a NFD,
- `lon_nfc`, `lon_nfd`: la longitud de las dos versiones,
- `iguales_directo`: si `PALABRA_NFC == PALABRA_NFD` (sin normalizar),
- `iguales_normalizando`: si son iguales tras normalizar **ambas** a NFC,
- `n_combinantes`: cuántos caracteres de categoría `"Mn"` (marcas combinantes) tiene `PALABRA_NFD`.

Antes de programar, predice a mano los cinco resultados.

In [ ]:
cadena_nfd = None            # TODO: PALABRA_NFC en forma NFD
lon_nfc = None               # TODO
lon_nfd = None               # TODO
iguales_directo = None       # TODO: comparación sin normalizar
iguales_normalizando = None  # TODO: compara tras normalizar las dos a NFC
n_combinantes = None         # TODO: usa unicodedata.category(c) == "Mn"

# Comprobación automática
assert all(v is not None for v in (cadena_nfd, lon_nfc, lon_nfd, iguales_directo, iguales_normalizando, n_combinantes)), "Completa las líneas marcadas con TODO"
assert cadena_nfd == PALABRA_NFD, "NFD descompone la ó en o + acento combinante"
assert (lon_nfc, lon_nfd) == (7, 8), "la versión NFD tiene un punto de código más"
assert iguales_directo is False and iguales_normalizando is True, "distintas sin normalizar, iguales al normalizar"
assert n_combinantes == 1, "solo hay una marca combinante: el acento"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Una función `limpiar` con expresiones regulares

Completa los cuatro patrones (`PATRON_URL`, `PATRON_MENCION_HASHTAG`, `PATRON_ALARGADA`, `PATRON_ESPACIO_ANTES`) para que `limpiar_simple` haga lo siguiente:

1. quitar URLs (`http://`, `https://` o `www.`) hasta el siguiente espacio,
2. quitar `@menciones` y `#hashtags`,
3. reducir a una las **letras** repetidas tres o más veces (¡nunca las cifras!: `1000` debe quedar como está); el patrón debe **capturar** la letra en un grupo,
4. quitar los espacios que quedan **antes** de `, ; : . ! ?` (capturando el signo en un grupo).

El resto de los pasos (reducir `!!!`, `....` y espacios múltiples) ya está escrito. `CASOS_LIMPIEZA` tiene cinco pares (entrada, resultado esperado), calculables a mano.

In [ ]:
PATRON_URL = None                # TODO: http(s):// o www. seguido de no-espacios
PATRON_MENCION_HASHTAG = None    # TODO: @ o # seguido de caracteres de palabra
PATRON_ALARGADA = None           # TODO: una letra (no cifra ni _) repetida 3 o más veces, capturada en un grupo
PATRON_ESPACIO_ANTES = None      # TODO: espacios seguidos de uno de , ; : . ! ? (capturado en un grupo)

def limpiar_simple(texto):
    texto = re.sub(PATRON_URL, " ", texto)
    texto = re.sub(PATRON_MENCION_HASHTAG, " ", texto)
    texto = re.sub(PATRON_ALARGADA, r"\1", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    texto = re.sub(r"\.{4,}", "...", texto)
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(PATRON_ESPACIO_ANTES, r"\1", texto)
    return texto.strip()

# Comprobación automática
assert all(p is not None for p in (PATRON_URL, PATRON_MENCION_HASHTAG, PATRON_ALARGADA, PATRON_ESPACIO_ANTES)), "Completa las líneas marcadas con TODO"
for entrada, esperado in CASOS_LIMPIEZA:
    assert limpiar_simple(entrada) == esperado, f"{entrada!r} → {limpiar_simple(entrada)!r} (esperado {esperado!r})"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Un tokenizador con `re.findall`

Escribe `PATRON_TOKEN` para que `tokenizar` separe:

- **números** con separadores decimales o de millares como una sola pieza (`4,99`, `12.5`, `1.250,50`),
- la **elisión** del italiano: una palabra seguida de apóstrofo y otra letra queda como `l'`, `un'`, `dell'`, y luego viene la palabra siguiente,
- **palabras** (letras y cifras `\w+`),
- cualquier otro carácter suelto (puntuación, `¿`, `¡`) como token propio.

Prueba el orden de las alternativas: las más específicas primero. `CASOS_TOKEN` tiene cinco frases con el resultado esperado.

In [ ]:
PATRON_TOKEN = None      # TODO: alternativas separadas por |: número | elisión | palabra | otro símbolo

def tokenizar(texto):
    return re.findall(PATRON_TOKEN, texto)

# Comprobación automática
assert PATRON_TOKEN is not None, "Completa las líneas marcadas con TODO"
for frase, esperado in CASOS_TOKEN:
    assert tokenizar(frase) == esperado, f"{frase!r} → {tokenizar(frase)}"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Tokens, tipos y hapax

Completa `resumir(tokens)` para que devuelva la cuádrupla `(n_tokens, n_tipos, n_hapax, top)`, donde `top` es la pareja `(palabra, frecuencia)` del tipo más frecuente. Se prueba con dos frases cortas ya tokenizadas (`TOKENS_ES` y `TOKENS_IT`), que puedes contar a mano.

In [ ]:
def resumir(tokens):
    conteo = Counter(tokens)
    n_tokens = None     # TODO: número de ocurrencias
    n_tipos = None      # TODO: número de palabras distintas
    n_hapax = None      # TODO: tipos que aparecen exactamente una vez
    top = None          # TODO: (palabra, frecuencia) del tipo más frecuente
    return n_tokens, n_tipos, n_hapax, top

# Comprobación automática
assert resumir(TOKENS_ES)[0] is not None, "Completa las líneas marcadas con TODO"
assert resumir(TOKENS_ES) == (12, 8, 5, ("el", 3)), f"revisa el recuento en la frase en español: {resumir(TOKENS_ES)}"   # dato-es
assert resumir(TOKENS_IT) == (11, 8, 6, ("il", 3)), f"revisa el recuento en la frase en italiano: {resumir(TOKENS_IT)}"
assert round(resumir(TOKENS_ES)[1] / resumir(TOKENS_ES)[0], 3) == 0.667, "TTR = tipos / tokens = 8 / 12"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Evaluar una limpieza contra el gold

`PARES` tiene 20 pares `(texto bruto, texto limpio de referencia)` en español y en italiano. Completa:

- `f1_tokens(pred, gold)`: F1 entre dos listas de tokens como **multiconjuntos** (comunes = suma de mínimos de las frecuencias; precisión = comunes / len(pred); *recall* = comunes / len(gold); 0 si no hay comunes),
- `evaluar(limpiador, pares)`: devuelve `(exacta, f1_medio)`, donde `exacta` es la proporción de pares cuyo resultado, tras `normalizar_comparacion`, coincide con el gold, y `f1_medio` la media del F1 de las palabras (`re.findall(r"\w+", ...)` en minúsculas).

Se comparan dos limpiadores: `sin_limpiar` y `limpiador_base` (quita URLs, menciones, hashtags y reduce `!!`).

> **Nota:** con 20 pares elegidos a mano solo se practica la lógica de la evaluación; no es una evaluación real.

In [ ]:
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto).casefold()).strip()

def sin_limpiar(texto):
    return texto

def limpiador_base(texto):
    texto = re.sub(r"(?:https?://|www\.)\S+", " ", texto)
    texto = re.sub(r"[@#]\w+", " ", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def f1_tokens(pred, gold):
    comunes = None       # TODO: suma de los mínimos de las frecuencias (Counter & Counter)
    if not comunes:
        return 0.0
    precision = None     # TODO
    recall = None        # TODO
    return 2 * precision * recall / (precision + recall)

def evaluar(limpiador, pares):
    aciertos, f1s = [], []
    for bruto, gold in pares:
        pred = normalizar_comparacion(limpiador(bruto))
        gold_n = normalizar_comparacion(gold)
        aciertos.append(None)    # TODO: ¿coincide pred con gold_n?
        f1s.append(f1_tokens(re.findall(r"\w+", pred), re.findall(r"\w+", gold_n)))
    return sum(aciertos) / len(pares), sum(f1s) / len(pares)

# Comprobación automática
assert f1_tokens(["a", "b", "c"], ["a", "b", "d"]) is not None, "Completa las líneas marcadas con TODO"
assert abs(f1_tokens(["a", "b", "c"], ["a", "b", "d"]) - 2 / 3) < 1e-9, "2 comunes de 3 y de 3: F1 = 2/3"
assert abs(f1_tokens(["a", "a"], ["a"]) - 2 / 3) < 1e-9, "el multiconjunto cuenta 1 común: precisión 1/2, recall 1"
assert f1_tokens([], ["a"]) == 0.0
exacta_sin, f1_sin = evaluar(sin_limpiar, PARES)
exacta_base, f1_base = evaluar(limpiador_base, PARES)
assert abs(exacta_sin - 0.4) < 1e-9 and abs(exacta_base - 0.6) < 1e-9, f"exacta: {exacta_sin}, {exacta_base}"
assert f1_base > f1_sin and f1_base > 0.9, f"F1: {f1_sin:.3f}, {f1_base:.3f}"
print(f"Ejercicio 5 correcto ✔ (exacta {exacta_sin:.2f} → {exacta_base:.2f}; F1 {f1_sin:.3f} → {f1_base:.3f})")

---
## Reto final — `preparar(texto, idioma)`

Integra lo visto: escribe `preparar(texto, idioma)`, que recibe un texto bruto y devuelve la **lista de tokens de palabra en minúsculas** (sin puntuación). Puedes reutilizar `limpiar_simple` y `tokenizar` de los ejercicios 2 y 3, pero el conjunto de prueba `PRUEBA_RETO` (ocho textos, cuatro por idioma, escritos para este reto) incluye cosas que esos ejercicios no cubren: **emojis**, **marcas `_cursiva_`** y **abreviaturas de chat** (`ABREVIATURAS[idioma]` te da el diccionario del idioma).

Se evalúa con el **F1 medio de tokens** y se exige un mínimo de 0,97.

> **Nota:** los pasos y la lista de abreviaturas se han elegido mirando estas mismas ocho frases: sirve para practicar, **no como evaluación real**. Con textos nuevos el F1 sería menor; una evaluación honesta usa frases que no has mirado al diseñar la limpieza.

In [ ]:
def preparar(texto, idioma):
    # TODO 1: quita emojis y marcas _cursiva_ (los ejercicios 2 y 3 no las cubren)
    # TODO 2: limpia, pasa a minúsculas y tokeniza
    # TODO 3: sustituye las abreviaturas con ABREVIATURAS[idioma] y deja solo tokens de palabra
    return None

# Comprobación automática
def f1_reto(pred, gold):
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if comunes == 0:
        return 0.0
    p, r = comunes / len(pred), comunes / len(gold)
    return 2 * p * r / (p + r)

f1s = []
for idioma, texto, esperado in PRUEBA_RETO:
    tokens = preparar(texto, idioma)
    assert tokens is not None, "Completa las líneas marcadas con TODO"
    f1s.append(f1_reto(tokens, esperado))
f1_medio = sum(f1s) / len(f1s)
assert f1_medio >= 0.97, f"F1 medio {f1_medio:.3f}; peores casos: {[i for i, f in enumerate(f1s) if f < 0.9]}"
print(f"Reto final correcto ✔ (F1 medio {f1_medio:.3f} sobre {len(f1s)} textos)")

**Tus conclusiones (escribe 3-4 frases):** ¿qué caso del reto te costó más y por qué? ¿Qué le pasaría a tu `preparar` con una errata (`esrellas`), con una palabra acentuada sin acento o con una abreviatura que no está en el diccionario? ¿Qué información has tirado al limpiar (emojis, mayúsculas, `!!!`) y en qué tarea te haría falta?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
cadena_nfd = unicodedata.normalize("NFD", PALABRA_NFC)
lon_nfc = len(PALABRA_NFC)
lon_nfd = len(PALABRA_NFD)
iguales_directo = PALABRA_NFC == PALABRA_NFD
iguales_normalizando = unicodedata.normalize("NFC", PALABRA_NFC) == unicodedata.normalize("NFC", PALABRA_NFD)
n_combinantes = sum(unicodedata.category(c) == "Mn" for c in PALABRA_NFD)

# Comprobación automática
assert all(v is not None for v in (cadena_nfd, lon_nfc, lon_nfd, iguales_directo, iguales_normalizando, n_combinantes)), "Completa las líneas marcadas con TODO"
assert cadena_nfd == PALABRA_NFD, "NFD descompone la ó en o + acento combinante"
assert (lon_nfc, lon_nfd) == (7, 8), "la versión NFD tiene un punto de código más"
assert iguales_directo is False and iguales_normalizando is True, "distintas sin normalizar, iguales al normalizar"
assert n_combinantes == 1, "solo hay una marca combinante: el acento"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
PATRON_URL = r"(?:https?://|www\.)\S+"
PATRON_MENCION_HASHTAG = r"[@#]\w+"
PATRON_ALARGADA = r"([^\W\d_])\1{2,}"
PATRON_ESPACIO_ANTES = r"\s+([,;:.!?])"

def limpiar_simple(texto):
    texto = re.sub(PATRON_URL, " ", texto)
    texto = re.sub(PATRON_MENCION_HASHTAG, " ", texto)
    texto = re.sub(PATRON_ALARGADA, r"\1", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    texto = re.sub(r"\.{4,}", "...", texto)
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(PATRON_ESPACIO_ANTES, r"\1", texto)
    return texto.strip()

# Comprobación automática
assert all(p is not None for p in (PATRON_URL, PATRON_MENCION_HASHTAG, PATRON_ALARGADA, PATRON_ESPACIO_ANTES)), "Completa las líneas marcadas con TODO"
for entrada, esperado in CASOS_LIMPIEZA:
    assert limpiar_simple(entrada) == esperado, f"{entrada!r} → {limpiar_simple(entrada)!r} (esperado {esperado!r})"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
PATRON_TOKEN = r"\d+(?:[.,]\d+)*|\w+'(?=\w)|\w+|[^\w\s]"

def tokenizar(texto):
    return re.findall(PATRON_TOKEN, texto)

# Comprobación automática
assert PATRON_TOKEN is not None, "Completa las líneas marcadas con TODO"
for frase, esperado in CASOS_TOKEN:
    assert tokenizar(frase) == esperado, f"{frase!r} → {tokenizar(frase)}"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def resumir(tokens):
    conteo = Counter(tokens)
    n_tokens = len(tokens)
    n_tipos = len(conteo)
    n_hapax = sum(1 for v in conteo.values() if v == 1)
    top = conteo.most_common(1)[0]
    return n_tokens, n_tipos, n_hapax, top

# Comprobación automática
assert resumir(TOKENS_ES)[0] is not None, "Completa las líneas marcadas con TODO"
assert resumir(TOKENS_ES) == (12, 8, 5, ("el", 3)), f"revisa el recuento en la frase en español: {resumir(TOKENS_ES)}"   # dato-es
assert resumir(TOKENS_IT) == (11, 8, 6, ("il", 3)), f"revisa el recuento en la frase en italiano: {resumir(TOKENS_IT)}"
assert round(resumir(TOKENS_ES)[1] / resumir(TOKENS_ES)[0], 3) == 0.667, "TTR = tipos / tokens = 8 / 12"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto).casefold()).strip()

def sin_limpiar(texto):
    return texto

def limpiador_base(texto):
    texto = re.sub(r"(?:https?://|www\.)\S+", " ", texto)
    texto = re.sub(r"[@#]\w+", " ", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def f1_tokens(pred, gold):
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if not comunes:
        return 0.0
    precision = comunes / len(pred)
    recall = comunes / len(gold)
    return 2 * precision * recall / (precision + recall)

def evaluar(limpiador, pares):
    aciertos, f1s = [], []
    for bruto, gold in pares:
        pred = normalizar_comparacion(limpiador(bruto))
        gold_n = normalizar_comparacion(gold)
        aciertos.append(pred == gold_n)
        f1s.append(f1_tokens(re.findall(r"\w+", pred), re.findall(r"\w+", gold_n)))
    return sum(aciertos) / len(pares), sum(f1s) / len(pares)

# Comprobación automática
assert f1_tokens(["a", "b", "c"], ["a", "b", "d"]) is not None, "Completa las líneas marcadas con TODO"
assert abs(f1_tokens(["a", "b", "c"], ["a", "b", "d"]) - 2 / 3) < 1e-9, "2 comunes de 3 y de 3: F1 = 2/3"
assert abs(f1_tokens(["a", "a"], ["a"]) - 2 / 3) < 1e-9, "el multiconjunto cuenta 1 común: precisión 1/2, recall 1"
assert f1_tokens([], ["a"]) == 0.0
exacta_sin, f1_sin = evaluar(sin_limpiar, PARES)
exacta_base, f1_base = evaluar(limpiador_base, PARES)
assert abs(exacta_sin - 0.4) < 1e-9 and abs(exacta_base - 0.6) < 1e-9, f"exacta: {exacta_sin}, {exacta_base}"
assert f1_base > f1_sin and f1_base > 0.9, f"F1: {f1_sin:.3f}, {f1_base:.3f}"
print(f"Ejercicio 5 correcto ✔ (exacta {exacta_sin:.2f} → {exacta_base:.2f}; F1 {f1_sin:.3f} → {f1_base:.3f})")

### Solución del reto final

In [ ]:
def preparar(texto, idioma):
    texto = re.sub(r"[\U0001F300-\U0001FAFF☀-➿]+", " ", texto)
    texto = re.sub(r"(?<!\w)_(\S(?:.*?\S)?)_(?!\w)", r"\1", texto)
    tokens = tokenizar(limpiar_simple(texto).lower())
    tokens = [ABREVIATURAS[idioma].get(t, t) for t in tokens]
    return [t for t in tokens if re.match(r"\w", t)]

# Comprobación automática
def f1_reto(pred, gold):
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if comunes == 0:
        return 0.0
    p, r = comunes / len(pred), comunes / len(gold)
    return 2 * p * r / (p + r)

f1s = []
for idioma, texto, esperado in PRUEBA_RETO:
    tokens = preparar(texto, idioma)
    assert tokens is not None, "Completa las líneas marcadas con TODO"
    f1s.append(f1_reto(tokens, esperado))
f1_medio = sum(f1s) / len(f1s)
assert f1_medio >= 0.97, f"F1 medio {f1_medio:.3f}; peores casos: {[i for i, f in enumerate(f1s) if f < 0.9]}"
print(f"Reto final correcto ✔ (F1 medio {f1_medio:.3f} sobre {len(f1s)} textos)")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (NFC/NFD, `casefold`, regex, tokens y tipos, evaluación de una limpieza)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*